# PyTorch Collate Functions and Transformations

This notebook demonstrates collate functions, transformations, and Compose with practical ML examples.


In [23]:
from typing import Any, List, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from torch import tensor
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from torchvision.transforms import Compose as TorchvisionCompose


## 1. Custom Collate Functions

Collate functions define how individual samples are combined into batches. Useful for handling variable-length sequences or custom batching logic.


### Custom Collate Function Examples (Three Steps)

- **Step 1: CustomDataset** — Shows a dataset with variable-length samples (e.g., sequences of different lengths).
- **Step 2: CustomDatasetFixLen** — Shows a dataset where all samples are padded/truncated to a fixed length for batching.
- **Step 3: dynamic_length_collate** — Shows a collate function that dynamically pads sequences in a batch, allowing flexible batching for models like RNNs.

#### Step 1: CustomDataset (variable-length samples)
- Each sample is a sequence of random length (e.g., simulating text or time series).
- Useful for tasks where input length varies (NLP, sequence modeling).
- Shows why default DataLoader batching may not work for such data.

In [24]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.xs = [
            list(range(11, 13)),
            list(range(13, 16)),
            list(range(16, 21)),
            list(range(21, 24)),
            list(range(22, 25)),
            list(range(25, 30)),
        ]
        self.ys = [0, 0, 0, 1, 1, 1]
        assert len(self.xs) == len(self.ys)
    def __len__(self): 
        return len(self.xs)
    def __getitem__(self, idx):
        return {
            "x": self.xs[idx],
            "y": self.ys[idx],
        }

In [25]:
dset = CustomDataset()
for item in dset:
    print(item)

{'x': [11, 12], 'y': 0}
{'x': [13, 14, 15], 'y': 0}
{'x': [16, 17, 18, 19, 20], 'y': 0}
{'x': [21, 22, 23], 'y': 1}
{'x': [22, 23, 24], 'y': 1}
{'x': [25, 26, 27, 28, 29], 'y': 1}


#### Step 2: CustomDatasetFixLen (fixed-length samples)
- Each sample is padded or truncated to a fixed length (e.g., for models that require fixed-size input).
- Useful for simple batching and when model architecture expects fixed input size.
- Shows how to preprocess variable-length data for compatibility with default DataLoader batching.

In [26]:
class CustomDatasetFixLen(torch.utils.data.Dataset):
    def __init__(self, max_len=10):
        self.max_len = max_len
        self.xs = [
            list(range(11, 13)),
            list(range(13, 16)),
            list(range(16, 21)),
            list(range(21, 24)),
            list(range(22, 25)),
            list(range(25, 30)),
        ]
        self.ys = [0, 0, 0, 1, 1, 1]
        assert len(self.xs) == len(self.ys)
    def __len__(self): 
        return len(self.xs)
    def __getitem__(self, idx):
        x = self.xs[idx]
        pad_len = self.max_len - len(x)
        x = x + [0]*pad_len
        return {
            "x": np.array(x),
            "y": self.ys[idx],
        }

In [27]:
dset = CustomDatasetFixLen(max_len=10)
for item in dset:
    print(item)

{'x': array([11, 12,  0,  0,  0,  0,  0,  0,  0,  0]), 'y': 0}
{'x': array([13, 14, 15,  0,  0,  0,  0,  0,  0,  0]), 'y': 0}
{'x': array([16, 17, 18, 19, 20,  0,  0,  0,  0,  0]), 'y': 0}
{'x': array([21, 22, 23,  0,  0,  0,  0,  0,  0,  0]), 'y': 1}
{'x': array([22, 23, 24,  0,  0,  0,  0,  0,  0,  0]), 'y': 1}
{'x': array([25, 26, 27, 28, 29,  0,  0,  0,  0,  0]), 'y': 1}


#### Step 3: dynamic_length_collate (collate function for dynamic padding)
- Pads all sequences in a batch to the length of the longest sequence in that batch.
- Enables batching of variable-length data without forcing all samples to a fixed length.
- Essential for RNNs, transformers, and models that can handle dynamic input sizes.

In [28]:
def dynamic_length_collate(batch):
    max_len = max(len(item["x"]) for item in batch)
    batch_x = []
    for item in batch:
        pad_len = max_len - len(item["x"])
        batch_x.append(item["x"] + [0]*pad_len)
    return {
        "x": tensor(batch_x).type(torch.float),
        "y": tensor([item["y"] for item in batch])
    }

In [29]:
dset = CustomDataset()  # Use our original dataset, without fix max_len
dloader = DataLoader(dset, batch_size=2, shuffle=False,
                     collate_fn=dynamic_length_collate)
for batch in dloader:
    print(batch)

{'x': tensor([[11., 12.,  0.],
        [13., 14., 15.]]), 'y': tensor([0, 0])}
{'x': tensor([[16., 17., 18., 19., 20.],
        [21., 22., 23.,  0.,  0.]]), 'y': tensor([0, 1])}
{'x': tensor([[22., 23., 24.,  0.,  0.],
        [25., 26., 27., 28., 29.]]), 'y': tensor([1, 1])}


## 2. Basic Transformations


* **Custom transformations** : Custom callable classes for simple preprocessing (normalization, standardization, noise).

* **Built-in (torchvision) transformations** : Common, optimized transforms provided by torchvision (e.g. ToTensor, Normalize).


### Custom transformation classes

In [ ]:
# Custom transformation classes
class Normalize:
    """Normalize data to [0, 1] range using min-max scaling."""

    def __init__(self, min_val=None, max_val=None):
        self.min_val = min_val
        self.max_val = max_val

    def __call__(self, x):
        if self.min_val is None or self.max_val is None:
            min_val = x.min()
            max_val = x.max()
        else:
            min_val = self.min_val
            max_val = self.max_val

        return (x - min_val) / (max_val - min_val + 1e-8)


class Standardize:
    """Standardize data to zero mean and unit variance."""

    def __init__(self, mean=None, std=None):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        if self.mean is None or self.std is None:
            mean = x.mean()
            std = x.std()
        else:
            mean = self.mean
            std = self.std

        return (x - mean) / (std + 1e-8)



In [ ]:
# Test individual transformations (concise)
sample_data = torch.randn(10, 3) * 5 + 10  # Mean ~10, std ~5
standardize = Standardize()
standardized_data = standardize(sample_data)
print("After standardization: mean={:.3f}, std={:.3f}".format(standardized_data.mean().item(), standardized_data.std().item()))

Original data:
  Mean: 11.485, Std: 4.825
  Min: 3.103, Max: 21.071


NameError: name 'Normalize' is not defined

### Built-in (torchvision) transformations

In [ ]:
# Short explanations
# - T.ToTensor: converts HxWxC numpy arrays or PIL images to CxHxW float tensors and scales pixel values to [0.0, 1.0].
# - T.Normalize: normalizes a tensor image with mean and std (per-channel).

# Example pipeline using torchvision transforms
torch_pipeline = T.Compose([
    T.ToTensor(),  # converts and scales to [0, 1]
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # scales to [-1, 1]
])

# Simulate image-like numpy data (HxWxC in 0-255 range)
img = (np.random.rand(32, 32, 3) * 255).astype(np.uint8)

# Apply transforms
img_tensor = torch_pipeline(img)
print('Transformed image tensor shape:', img_tensor.shape)
print('Tensor range after Normalize approx: mean {:.3f}, std {:.3f}'.format(img_tensor.mean().item(), img_tensor.std().item()))

## 3. Compose - Chaining Transformations

There are multiple ways to chain transformations in PyTorch. We'll explore three approaches: custom Compose, torchvision.transforms.Compose, and torch.nn.Sequential.


### Compose Approaches Summary

- **Custom Compose**:

  - Maximum flexibility, works with any callable
  - Best for: Custom transformations, research, prototyping

- **torchvision.transforms.Compose**:

  - Standard in computer vision workflows
  - Best for: CV pipelines, torchscript compatibility, established workflows

- **torch.nn.Sequential**:

  - Integrates with PyTorch's neural network ecosystem
  - Best for: GPU optimization, train/eval modes, differentiable transforms


### Custom Compose

In [ ]:
# Approach 1: Custom Compose implementation
class Compose:
    """Custom compose implementation - most flexible, works with any callable."""

    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, x):
        for transform in self.transforms:
            x = transform(x)
        return x


# Test data
test_data = torch.randn(5, 4) * 2 + 5
print("Original data:")
print(f"  Mean: {test_data.mean():.3f}, Std: {test_data.std():.3f}")

# Create pipeline with custom Compose
custom_pipeline = Compose(
    [Standardize(), Normalize(min_val=-3, max_val=3)]
)

custom_result = custom_pipeline(test_data)
print(f"\nCustom Compose result:")
print(f"  Mean: {custom_result.mean():.3f}, Std: {custom_result.std():.3f}")
print("  ✓ Best for: Maximum flexibility, any callable functions")

### torchvision.transforms.Compose

In [ ]:
# Approach 2: torchvision.transforms.Compose
torchvision_pipeline = TorchvisionCompose(
    [Standardize(), Normalize(min_val=-3, max_val=3)]
)

torchvision_result = torchvision_pipeline(test_data)
print(f"\nTorchvision Compose result:")
print(f"  Mean: {torchvision_result.mean():.3f}, Std: {torchvision_result.std():.3f}")
print("  ✓ Best for: Standard computer vision pipelines, torchscript compatibility")

### torch.nn.Sequential

In [ ]:
# Approach 3: torch.nn.Sequential (for nn.Module transformations)
# First, create nn.Module versions of our transformations


class StandardizeModule(torch.nn.Module):
    """nn.Module version of Standardize transform."""

    def __init__(self, mean=None, std=None):
        super().__init__()
        self.mean = mean
        self.std = std

    def forward(self, x):
        if self.mean is None or self.std is None:
            mean = x.mean()
            std = x.std()
        else:
            mean = self.mean
            std = self.std
        return (x - mean) / (std + 1e-8)


# Create pipeline with Sequential
sequential_pipeline = torch.nn.Sequential(
    StandardizeModule()
)

# Test in training mode
sequential_pipeline.train()
sequential_result_train = sequential_pipeline(test_data)
print(f"\nSequential (training mode) result:")
print(f"  Mean: {sequential_result_train.mean():.3f}, Std: {sequential_result_train.std():.3f}")

# Test in eval mode (no noise)
sequential_pipeline.eval()
sequential_result_eval = sequential_pipeline(test_data)
print(f"\nSequential (eval mode) result:")
print(f"  Mean: {sequential_result_eval.mean():.3f}, Std: {sequential_result_eval.std():.3f}")
print("  ✓ Best for: Neural network modules, GPU optimization, train/eval modes")

## 4. Dataset with Transformations

Integrating transformations with Dataset and DataLoader for complete preprocessing pipeline.


In [ ]:
# Dataset that applies transformations
class TransformDataset(Dataset):
    def __init__(self, size=100, transform=None):
        # Generate synthetic tabular data
        self.data = torch.randn(size, 5) * 3 + 2  # 5 features
        self.labels = torch.randint(0, 3, (size,))  # 3-class classification
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        label = self.labels[idx]

        # Apply transformations if provided
        if self.transform:
            sample = self.transform(sample)

        return sample, label


# Create transformation pipeline
transform_pipeline = TorchvisionCompose([Standardize()])

# Create dataset with transformations
dataset = TransformDataset(size=50, transform=transform_pipeline)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

# Show transformed data
batch_data, batch_labels = next(iter(loader))
print("Transformed data batch:")
print(f"  Shape: {batch_data.shape}")
print(f"  Mean: {batch_data.mean():.3f}, Std: {batch_data.std():.3f}")
print(f"  Labels: {batch_labels}")
print("\n✓ Transformations applied automatically in __getitem__")

## Summary

- **Collate Functions**: Custom batching logic for variable-length or complex data structures
- **Transformations**: Preprocessing operations like normalization, standardization, and augmentation
- **Compose**: Chain multiple transformations into a single pipeline
- **Dataset Integration**: Apply transformations seamlessly within Dataset classes